# Phase 20: Behavioral Feature Engineering

**Goal:** We have engineered Network Rate Features and Temporal Flow Features. Now, we must analyze the **Behavior** of an IP across multiple connections.

This is crucial for detecting **Lateral Movement** and **Zero-Day Attacks**. Even if the payload of the packet is encrypted (SSL/TLS), an attacker still has to behave suspiciously (e.g. scanning unique ports or generating anomalous volumes of traffic compared to the rest of the network)!

In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd  # type: ignore  # pylint: disable=import-error
import numpy as np  # type: ignore  # pylint: disable=import-error
from dataclasses import dataclass  # type: ignore  # pylint: disable=import-error
from scipy.stats import percentileofscore  # type: ignore  # pylint: disable=import-error

pd.set_option('display.max_columns', None)

### Step 1: The Behavioral Feature Registry
As always, we strictly document our new Mathematical Features so they can be parsed into Table 2 of your Research Paper.

In [2]:
@dataclass
class FeatureSpec:
    name: str
    description: str
    target_attack: str
    
BEHAVIORAL_FEATURES = []

### Step 2: Unique Destination Features (Subphase 20.1)
If a normal user connects to a server, they hit 1 IP and 1 Port (e.g., 443 for HTTPS). 
If a hacker is scanning your network, they will hit 100 unique IPs or 100 unique Ports in a matter of seconds. We need a rolling function to count `unique_dst_ports` and `unique_dst_ips`.

In [3]:
def calculate_unique_destinations(df: pd.DataFrame, target_col: str, window_size: int = 10) -> pd.Series:
    df = df.sort_values("timestamp")
    
    # Pandas rolling.apply() only works on numbers, not strings (like IP addresses).
    # We use pd.factorize to temporarily turn IPs/Strings into simple ID numbers so the math works!
    numeric_targets = pd.factorize(df[target_col])[0]
    df['temp_numeric_target'] = numeric_targets
    
    return df.groupby("source_ip")['temp_numeric_target'].rolling(window=window_size, min_periods=1).apply(lambda x: len(np.unique(x))).reset_index(level=0, drop=True)

BEHAVIORAL_FEATURES.extend([
    FeatureSpec("unique_dst_ports", "Count of unique destination ports in rolling window", "Port Scanning"),
    FeatureSpec("unique_dst_ips", "Count of unique destination IPs in rolling window", "Lateral Movement / Botnets")
])
print("✅ Unique Destination Analyzers Registered!")


✅ Unique Destination Analyzers Registered!


### Step 3: Protocol Switch Rate (Subphase 20.1)
Normal users usually stick to a single protocol (like TCP for streaming). 
Hackers often rapidly alternate between TCP, UDP, and ICMP as they execute different hacking tools (e.g., Nmap scans). The `protocol_switch_rate` detects this erratic behavior!

In [4]:
def calculate_protocol_switch_rate(df: pd.DataFrame, window_size: int = 5) -> pd.Series:
    df = df.sort_values("timestamp")
    
    # 1. Check if the current protocol is DIFFERENT from the previous row's protocol
    # We use .shift(1) to peek at the previous packet
    df['is_switch'] = (df['protocol'] != df.groupby('source_ip')['protocol'].shift(1)).astype(float)
    
    # 2. The first packet of a session shouldn't count as a "switch"
    df.loc[df.groupby('source_ip')['protocol'].head(1).index, 'is_switch'] = 0.0
    
    # 3. Calculate the rolling percentage (mean) of switches
    switch_rate = df.groupby("source_ip")['is_switch'].rolling(window=window_size, min_periods=1).mean()
    return switch_rate.reset_index(level=0, drop=True)

BEHAVIORAL_FEATURES.append(
    FeatureSpec("protocol_switch_rate", "Fraction of consecutive connections where the protocol changes", "Evaded Scans / Botnets")
)
print("✅ Protocol Switch Rate Registered!")

✅ Protocol Switch Rate Registered!


### Step 4: Source IP Request Volume Percentile (Subphase 20.2)
A standard hacker might send 1,000 packets. Is that a lot? It depends!

If the network is idle, 1,000 packets makes them the #1 top traffic generator (Top 99th Percentile). If it's a huge corporate network, 1,000 packets might just be average (50th Percentile).

We use `scipy.stats.percentileofscore` to dynamically calculate exactly how suspicious a user's traffic volume is compared to everyone else on the network right now.

In [5]:
def calculate_volume_percentile(df: pd.DataFrame) -> pd.Series:
    # 1. Count how many total packets every IP on the network sent
    all_ip_counts = df['source_ip'].value_counts()
    
    # 2. For every row, find out what percentile that specific user is in
    def get_percentile(ip):
        user_count = all_ip_counts[ip]
        return percentileofscore(all_ip_counts.values, user_count)
        
    return df['source_ip'].apply(get_percentile)

BEHAVIORAL_FEATURES.append(
    FeatureSpec("request_volume_percentile", "Percentile rank of source IP volume vs all active IPs", "DDoS / Data Exfiltration")
)
print("✅ Volume Percentile Registered!")

✅ Volume Percentile Registered!


### Step 5: Behavioral Integration Test
Let's put this to the test! I will generate a simulated network with two normal users (IP `1.1.1.1` and `2.2.2.2`). They stick to their own protocols and only send a few packets. 

Then, a Hacker (`10.0.0.99`) connects. The hacker rapidly scans multiple ports, rapidly switches between TCP and UDP, and floods the network!

In [6]:
# 1. Create a simulated network
fake_network = pd.DataFrame({
    "timestamp": [1, 2, 3, 4, 5, 6, 7, 8], 
    "source_ip": [
        "1.1.1.1", "1.1.1.1", # Normal User 1
        "2.2.2.2", "2.2.2.2", # Normal User 2
        "10.0.0.99", "10.0.0.99", "10.0.0.99", "10.0.0.99" # The Hacker!
    ], 
    "dest_ip": ["8.8.8.8", "8.8.8.8", "9.9.9.9", "9.9.9.9", "1.2.3.4", "1.2.3.5", "1.2.3.6", "1.2.3.7"],
    "dest_port": [443, 443, 80, 80, 22, 23, 3389, 445],
    "protocol": ["TCP", "TCP", "TCP", "TCP", "TCP", "UDP", "TCP", "ICMP"]
})

print("=== RAW NETWORK DATA ===")
display(fake_network)

# 2. Apply the Behavioral Feature Engineering
df = fake_network.copy()
df['unique_dst_ports'] = calculate_unique_destinations(df, "dest_port")
df['unique_dst_ips'] = calculate_unique_destinations(df, "dest_ip")
df['protocol_switch_rate'] = calculate_protocol_switch_rate(df)
df['volume_percentile'] = calculate_volume_percentile(df)

print("\n=== ENGINEERED BEHAVIORAL FEATURES ===")
display(df)

print("\nNotice what happened to Hacker IP 10.0.0.99:")
print("- They hit 4 unique Ports AND 4 unique IPs! (Massive Lateral Movement flagged)")
print("- Their protocol switch rate spiked because they rapidly alternated between TCP, UDP, and ICMP!")
print("- Because they generated 50% of the network's total traffic, the AI mathematically calculates they are in the 100th (highest) percentile of volume!")

print("\n✅ BEHAVIORAL REGISTRY COMPLETE: ")
for f in BEHAVIORAL_FEATURES:
    print(f"- {f.name}: {f.description}")

=== RAW NETWORK DATA ===


,timestamp,source_ip,dest_ip,dest_port,protocol
0,1,1.1.1.1,8.8.8.8,443,TCP
1,2,1.1.1.1,8.8.8.8,443,TCP
2,3,2.2.2.2,9.9.9.9,80,TCP
3,4,2.2.2.2,9.9.9.9,80,TCP
4,5,10.0.0.99,1.2.3.4,22,TCP
5,6,10.0.0.99,1.2.3.5,23,UDP
6,7,10.0.0.99,1.2.3.6,3389,TCP
7,8,10.0.0.99,1.2.3.7,445,ICMP



=== ENGINEERED BEHAVIORAL FEATURES ===


,timestamp,source_ip,dest_ip,dest_port,protocol,unique_dst_ports,unique_dst_ips,protocol_switch_rate,volume_percentile
0,1,1.1.1.1,8.8.8.8,443,TCP,1.0,1.0,0.000000,50.0
1,2,1.1.1.1,8.8.8.8,443,TCP,1.0,1.0,0.000000,50.0
2,3,2.2.2.2,9.9.9.9,80,TCP,1.0,1.0,0.000000,50.0
3,4,2.2.2.2,9.9.9.9,80,TCP,1.0,1.0,0.000000,50.0
4,5,10.0.0.99,1.2.3.4,22,TCP,1.0,1.0,0.000000,100.0
5,6,10.0.0.99,1.2.3.5,23,UDP,2.0,2.0,0.500000,100.0
6,7,10.0.0.99,1.2.3.6,3389,TCP,3.0,3.0,0.666667,100.0
7,8,10.0.0.99,1.2.3.7,445,ICMP,4.0,4.0,0.750000,100.0



Notice what happened to Hacker IP 10.0.0.99:
- They hit 4 unique Ports AND 4 unique IPs! (Massive Lateral Movement flagged)
- Their protocol switch rate spiked because they rapidly alternated between TCP, UDP, and ICMP!
- Because they generated 50% of the network's total traffic, the AI mathematically calculates they are in the 100th (highest) percentile of volume!

✅ BEHAVIORAL REGISTRY COMPLETE: 
- unique_dst_ports: Count of unique destination ports in rolling window
- unique_dst_ips: Count of unique destination IPs in rolling window
- protocol_switch_rate: Fraction of consecutive connections where the protocol changes
- request_volume_percentile: Percentile rank of source IP volume vs all active IPs
